# Day 1 - Lab 1: Guided Walkthrough
## AI and Machine Learning Fundamentals
### Titanic Dataset

**What you will build:**
A complete ML pipeline — raw data, cleaning, 12 algorithms, evaluation metrics, and advanced techniques.

**How to use this lab:**
Read each explanation cell, then run the code cell below it.
Every line of code has a comment telling you what it does.


## Section 0: Setup
Run this cell first.

In [ ]:
pip install --upgrade jupyter_client jupyter_core ipykernel notebook

In [ ]:
# --- Standard library ---
import os
import warnings
import urllib.request

warnings.filterwarnings('ignore')   # hide unimportant warning messages

# --- Data and maths ---
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

# --- Preprocessing ---
from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- Models ---
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# --- Utilities ---
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)

print("All imports successful!")


## Load the Dataset

The cell below checks if you are on Kaggle or a local machine and loads the data from the right place.

In [ ]:
# Detect whether we are on Kaggle or a local machine
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    # On Kaggle: add the Titanic dataset via the sidebar
    # (Notebook sidebar -> + Add Data -> Competitions -> Titanic)
    TITANIC_PATH =  '/kaggle/input/datasets/hesh97/titanicdataset-traincsv/train.csv'
    print("Running on Kaggle")
else:
    # On a local machine: use the CSV in this folder
    TITANIC_PATH = 'titanic.csv'
    print("Running on a local machine")

    # If the file does not exist yet, download it automatically
    if not os.path.exists(TITANIC_PATH):
        print("Downloading titanic.csv ...")
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv',
            TITANIC_PATH
        )
        print("Download complete!")

# Load the CSV into a pandas DataFrame
df_raw = pd.read_csv(TITANIC_PATH)

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
df_raw.head()


## Section 1: Exploratory Data Analysis (EDA)

Before changing anything, we **look** at the data:
- How many rows and columns are there?
- What data type is each column?
- Which columns have missing values?
- What does the target column look like?

> **Curriculum link (Module 1):** Patterns only emerge when you see the data first.


In [ ]:
# Make a working copy so the original is never changed
df = df_raw.copy()

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print()
print("Column names and types:")
print(df.dtypes)
print()
print("First 5 rows:")
df.head()


In [ ]:
# Statistical summary (mean, min, max, etc.) for every number column
df.describe()


In [ ]:
# Count missing values per column
missing_counts = df.isnull().sum()
print("Missing values per column:")
print(missing_counts)


In [ ]:
# Plot 1: bar chart of missing values
# Plot 2: how many passengers survived vs died

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left plot — missing values
missing_only = df.isnull().sum()
missing_only = missing_only[missing_only > 0]   # keep only columns that have missing values
missing_only.plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('Missing values per column')
axes[0].set_ylabel('Count')

# Right plot — survival counts
survived_counts = df['Survived'].value_counts()
survived_counts.plot(kind='bar', ax=axes[1], color=['steelblue', 'darkorange'])
axes[1].set_title('Survived (0 = Died, 1 = Survived)')
axes[1].set_xticklabels(['Died', 'Survived'], rotation=0)

plt.tight_layout()
plt.show()

# Calculate and print the survival rate
survival_rate = df['Survived'].mean()
print("Overall survival rate:", round(survival_rate * 100, 1), "%")


In [ ]:
# Correlation heatmap — how much do columns move together?
# We only use numeric columns for this.
numeric_columns = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

plt.figure(figsize=(8, 5))
sns.heatmap(df[numeric_columns].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Between Features')
plt.tight_layout()
plt.show()


## Section 2: Data Cleaning (7 Steps)

> **Curriculum link (Module 2):** Garbage in, garbage out.

We follow the 7-step checklist:
1. Remove irrelevant features
2. Remove duplicates
3. Handle missing values
4. Encode categorical variables
5. Scale numerical features
6. Detect outliers
7. Feature engineering


In [ ]:
# Step 1: Remove irrelevant features
# - PassengerId is just a row number, not a real feature
# - Name and Ticket are unique strings that carry no pattern
# - Cabin has 77% of its values missing — too sparse to use

columns_to_remove = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df = df.drop(columns=columns_to_remove)

print("Removed:", columns_to_remove)
print("Remaining columns:", list(df.columns))


In [ ]:
# Step 2: Check for and remove duplicate rows
number_of_duplicates = df.duplicated().sum()
print("Number of duplicate rows:", number_of_duplicates)

if number_of_duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates removed. New row count:", df.shape[0])
else:
    print("No duplicates found — nothing to remove.")


In [ ]:
# Step 3: Handle missing values
print("Missing values BEFORE filling:")
print(df.isnull().sum())
print()

# Fill missing Age values with the median age
# (median is better than mean because it is not pulled by extreme values)
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)

# Fill missing Embarked values with the most common port
most_common_port = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(most_common_port)

print("Missing values AFTER filling:")
print(df.isnull().sum())


In [ ]:
# Step 4: Encode categorical variables
# Machine learning models only understand numbers, not words.

# Sex column: "male" -> 1, "female" -> 0
encoder = LabelEncoder()
df['Sex'] = encoder.fit_transform(df['Sex'])

# Embarked column: 3 categories -> 2 new 0/1 columns (one-hot encoding)
# drop_first=True removes one column to avoid redundancy
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

print("Column types after encoding:")
print(df.dtypes)
print()
df.head(3)


In [ ]:
# Step 5: Scale numerical features
# StandardScaler shifts every column so that:
#   mean = 0 and standard deviation = 1
# This stops large-valued columns (like Fare) from dominating the model.

scaler = StandardScaler()

columns_to_scale = ['Age', 'Fare', 'SibSp', 'Parch']
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

print("First 3 rows after scaling:")
df[columns_to_scale].head(3)


In [ ]:
# Step 6: Detect outliers with box plots
# A box plot shows the middle 50% of the data and any extreme values.

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].boxplot(df['Fare'].dropna(), vert=False)
axes[0].set_title('Fare (after scaling)')

axes[1].boxplot(df['Age'].dropna(), vert=False)
axes[1].set_title('Age (after scaling)')

plt.tight_layout()
plt.show()

print("The dots beyond the whiskers are outliers.")
print("Tree-based models (Decision Tree, Random Forest) handle outliers naturally.")


In [ ]:
# Step 7: Feature engineering — create new, useful columns from existing ones

# FamilySize: total number of family members on board
df['FamilySize'] = df['SibSp'] + df['Parch']

# IsAlone: was this passenger travelling alone? (1 = yes, 0 = no)
df['IsAlone'] = 0                          # start with 0 for everyone
df.loc[df['FamilySize'] == 0, 'IsAlone'] = 1   # set to 1 where family is 0

print("New columns added: FamilySize and IsAlone")
print()
print("IsAlone value counts:")
print(df['IsAlone'].value_counts())
print()
df.head(3)


## Section 3: Machine Learning Setup

> **Curriculum link (Module 3):** Supervised learning — we have labelled data (Survived = 0 or 1).

**Why three sets?**
- **Train set (70%)** — the model learns from this
- **Validation set (15%)** — we tune the model on this WITHOUT touching the test set
- **Test set (15%)** — we evaluate the final model ONCE on this at the very end


In [ ]:
# Separate features (X) from the target label (y)
X = df.drop(columns=['Survived'])   # everything except Survived
y = df['Survived']                  # what we want to predict

# Step 1: hold out 15% for the test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.15,     # 15% goes to test
    random_state=42,    # makes the split reproducible
    stratify=y          # keeps the same survival ratio in each set
)

# Step 2: split the remaining 85% into 70% train and 15% validation
# 15/85 = 0.176, so test_size=0.176 gives us the right proportion
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.176,
    random_state=42,
    stratify=y_trainval
)

print("Train rows:     ", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:      ", len(X_test))
print()
print("Features used:", list(X.columns))


## Section 4: The 12 Algorithms

We implement all 12 algorithms from Module 4.
Each section follows the same 4-step pattern:

```
Step 1: Create the model
Step 2: Train the model  (model.fit)
Step 3: Make predictions (model.predict)
Step 4: Evaluate the results
```


### Algorithm 4.1: Linear Regression

**Type:** Supervised — Regression
**Idea:** Draw the best-fit straight line through the data points.
The formula is: prediction = (weight * feature) + bias
**Use when:** You want to predict a number (price, temperature, age).

We use the California Housing dataset here because it is a pure regression problem
(predicting house prices).


In [ ]:
# Load the California Housing dataset (comes with scikit-learn, no download needed)
housing = fetch_california_housing(as_frame=True)
X_housing = housing.data     # the features
y_housing = housing.target   # the target: median house price (in $100 000s)

# Split into train and test
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

# Scale the features (linear regression is sensitive to scale)
scaler_h = StandardScaler()
X_h_train_scaled = scaler_h.fit_transform(X_h_train)
X_h_test_scaled  = scaler_h.transform(X_h_test)

# --- Step 1: Create the model ---
lr_model = LinearRegression()

# --- Step 2: Train the model ---
lr_model.fit(X_h_train_scaled, y_h_train)

# --- Step 3: Make predictions ---
y_h_predictions = lr_model.predict(X_h_test_scaled)

# --- Step 4: Evaluate ---
mse  = mean_squared_error(y_h_test, y_h_predictions)
mae  = mean_absolute_error(y_h_test, y_h_predictions)
rmse = mse ** 0.5           # RMSE = square root of MSE
r2   = r2_score(y_h_test, y_h_predictions)

print("Linear Regression results (California Housing):")
print("  RMSE:", round(rmse, 4), "  (in $100 000 units — lower is better)")
print("  MAE :", round(mae, 4))
print("  R2  :", round(r2, 4),  "  (1.0 = perfect, 0.0 = no better than guessing the mean)")

# Plot: actual values vs predicted values
plt.figure(figsize=(6, 4))
plt.scatter(y_h_test[:300], y_h_predictions[:300], alpha=0.4, s=15)
min_val = y_h_test.min()
max_val = y_h_test.max()
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect prediction')
plt.xlabel('Actual price')
plt.ylabel('Predicted price')
plt.title('Linear Regression: Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()


### Algorithm 4.2: Logistic Regression

**Type:** Supervised — Classification
**Idea:** Like linear regression, but wraps the result in a sigmoid curve
so the output is always between 0 and 1 (a probability).
If P > 0.5 -> predict class 1; if P <= 0.5 -> predict class 0.
**Use when:** You want to classify something into two groups (survived or not, spam or not).


In [ ]:
# --- Step 1: Create the model ---
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# --- Step 2: Train on the training set ---
log_reg.fit(X_train, y_train)

# --- Step 3: Predict on the validation set ---
y_pred_logreg = log_reg.predict(X_val)

# --- Step 4: Evaluate ---
accuracy = accuracy_score(y_val, y_pred_logreg)
print("Logistic Regression — Validation Accuracy:", round(accuracy, 4))
print()
print(classification_report(y_val, y_pred_logreg, target_names=['Died', 'Survived']))


### Algorithm 4.3: K-Nearest Neighbors (KNN)

**Type:** Supervised — Classification or Regression
**Idea:** To classify a new point, look at the K closest points in the training data.
The new point gets the label that the majority of those K neighbours have.
**Key question:** What is the best value of K?
**Use when:** You have a small dataset and want a simple, intuitive model.


In [ ]:
# Find the best value of K by testing K = 1 to 20
k_scores = []

for k in range(1, 21):
    # Create a model with this value of K
    knn_model = KNeighborsClassifier(n_neighbors=k)
    # Train it
    knn_model.fit(X_train, y_train)
    # Evaluate on the validation set
    val_accuracy = accuracy_score(y_val, knn_model.predict(X_val))
    # Save the score
    k_scores.append(val_accuracy)

# Find which K gave the highest accuracy
best_accuracy = max(k_scores)
best_k = k_scores.index(best_accuracy) + 1   # +1 because our list starts at k=1

print("Best K:", best_k, "  Validation Accuracy:", round(best_accuracy, 4))

# Plot accuracy vs K
plt.figure(figsize=(8, 3))
plt.plot(range(1, 21), k_scores, marker='o', linewidth=2)
plt.axvline(x=best_k, color='red', linestyle='--', label='Best K = ' + str(best_k))
plt.xlabel('K (number of neighbours)')
plt.ylabel('Validation Accuracy')
plt.title('KNN: Choosing the Best K')
plt.legend()
plt.tight_layout()
plt.show()

# Train the final KNN model with the best K
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)


### Algorithm 4.4: Naive Bayes

**Type:** Supervised — Classification
**Idea:** Uses probability theory. For each class, it calculates the probability
that the data belongs to it, and picks the most likely class.
The "naive" part: it assumes all features are independent of each other.
**Use when:** Quick baseline, text classification, works well with small data.


In [ ]:
# --- Step 1: Create ---
nb_model = GaussianNB()

# --- Step 2: Train ---
nb_model.fit(X_train, y_train)

# --- Step 3: Predict ---
y_pred_nb = nb_model.predict(X_val)

# --- Step 4: Evaluate ---
nb_accuracy = accuracy_score(y_val, y_pred_nb)
print("Naive Bayes — Validation Accuracy:", round(nb_accuracy, 4))


### Algorithm 4.5: Support Vector Machine (SVM)

**Type:** Supervised — Classification or Regression
**Idea:** Find the boundary line (or hyperplane) that separates the two classes
with the largest possible gap (called the "margin").
Points closest to the boundary are called "support vectors".
The kernel trick lets SVM draw curved boundaries, not just straight lines.
**Use when:** The classes are clearly separable, or you have high-dimensional data (e.g. images).


In [ ]:
# --- Step 1: Create ---
# kernel='rbf' uses a curved (radial basis function) boundary
# probability=True lets us get probability outputs later (needed for ROC curves)
svm_model = SVC(kernel='rbf', probability=True, random_state=42)

# --- Step 2: Train ---
svm_model.fit(X_train, y_train)

# --- Step 3: Predict ---
y_pred_svm = svm_model.predict(X_val)

# --- Step 4: Evaluate ---
svm_accuracy = accuracy_score(y_val, y_pred_svm)
print("SVM (RBF kernel) — Validation Accuracy:", round(svm_accuracy, 4))


### Algorithm 4.6: Decision Tree

**Type:** Supervised — Classification or Regression
**Idea:** Ask yes/no questions about the features, one at a time.
Each question splits the data into two groups. This continues until
each group is mostly one class. You end up with a flowchart of rules.
**Pros:** Very easy to understand and visualise.
**Cons:** Tends to memorise the training data (overfitting).


In [ ]:
# --- Step 1: Create ---
# max_depth=4 limits how many levels of questions the tree can ask
# This prevents the tree from becoming too complex and overfitting
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)

# --- Step 2: Train ---
dt_model.fit(X_train, y_train)

# --- Step 3: Predict ---
y_pred_dt = dt_model.predict(X_val)

# --- Step 4: Evaluate ---
dt_accuracy = accuracy_score(y_val, y_pred_dt)
print("Decision Tree — Validation Accuracy:", round(dt_accuracy, 4))

# Visualise the tree (shows the actual rules it learned)
plt.figure(figsize=(18, 6))
plot_tree(
    dt_model,
    feature_names=list(X_train.columns),
    class_names=['Died', 'Survived'],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title('Decision Tree (max_depth = 4)')
plt.tight_layout()
plt.show()

# Which features did the tree find most useful?
feature_names = list(X_train.columns)
importance_values = dt_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance_values
})
importance_df = importance_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 3))
plt.bar(importance_df['Feature'], importance_df['Importance'])
plt.title('Decision Tree: Feature Importance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Algorithm 4.7: Random Forest

**Type:** Supervised — Classification or Regression
**Idea:** Train many decision trees (a "forest"), each on a random portion of
the data and a random subset of features.
For classification, the final answer is the majority vote of all trees.
**Why it beats a single tree:** Many imperfect trees together are more reliable
than one tree trying to do everything. This is called "bagging".


In [ ]:
# --- Step 1: Create ---
# n_estimators=100 means 100 trees in the forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

# --- Step 2: Train ---
rf_model.fit(X_train, y_train)

# --- Step 3: Predict ---
y_pred_rf = rf_model.predict(X_val)

# --- Step 4: Evaluate ---
rf_accuracy = accuracy_score(y_val, y_pred_rf)
print("Random Forest (100 trees) — Validation Accuracy:", round(rf_accuracy, 4))

# Feature importance
feature_names = list(X_train.columns)
importance_values = rf_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance_values
})
importance_df = importance_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 3))
plt.bar(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.title('Random Forest: Feature Importance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Algorithm 4.8: K-Means Clustering

**Type:** Unsupervised (no labels needed)
**Idea:**
1. Pick K starting points (called centroids) at random
2. Assign every data point to its nearest centroid
3. Move each centroid to the average position of its assigned points
4. Repeat steps 2-3 until nothing changes

**Use when:** You want to discover natural groups in your data without
knowing the groups in advance (customer segmentation, document clustering).


In [ ]:
# Use Age and Fare from the raw (original) data for an easy-to-see plot
cluster_data = df_raw[['Age', 'Fare']].dropna().copy()

# Scale the values so both columns have equal influence
scaler_c = StandardScaler()
cluster_data_scaled = scaler_c.fit_transform(cluster_data)

# Elbow method: try different values of K and plot the inertia
# Inertia = how tightly packed the clusters are (lower = better, but not always K=2 is best)
inertia_values = []
k_values = [2, 3, 4, 5, 6, 7, 8, 9]

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(cluster_data_scaled)
    inertia_values.append(km.inertia_)

plt.figure(figsize=(7, 3))
plt.plot(k_values, inertia_values, marker='o')
plt.xlabel('K (number of clusters)')
plt.ylabel('Inertia')
plt.title('Elbow Method: How to Choose K')
plt.tight_layout()
plt.show()

print("Look for the 'elbow' — the point where adding more clusters stops helping much.")

# Fit with K=3
km_model = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = km_model.fit_predict(cluster_data_scaled)

plt.figure(figsize=(7, 4))
plt.scatter(cluster_data['Age'], cluster_data['Fare'], c=cluster_labels,
            cmap='viridis', alpha=0.5, s=15)
plt.xlabel('Age')
plt.ylabel('Fare')
plt.title('K-Means (K=3): Passenger Clusters')
plt.tight_layout()
plt.show()


### Algorithm 4.9: Principal Component Analysis (PCA)

**Type:** Unsupervised — Dimensionality Reduction
**Idea:** Find the directions in the data along which the values spread out the most.
Then project all the data onto just a few of these directions.
This lets you visualise high-dimensional data in 2D or 3D.
**Use when:** You have many features and want to reduce noise, speed up training,
or visualise your data.


In [ ]:
# Reduce all features down to 2 "principal components" for easy visualisation
pca = PCA(n_components=2)

# Fit on the training data and transform it
X_pca = pca.fit_transform(X_train)

# How much of the total variance do these 2 components explain?
variance_ratio = pca.explained_variance_ratio_
total_variance = variance_ratio[0] + variance_ratio[1]

print("Variance explained by component 1:", round(variance_ratio[0] * 100, 1), "%")
print("Variance explained by component 2:", round(variance_ratio[1] * 100, 1), "%")
print("Total variance explained:         ", round(total_variance * 100, 1), "%")

# Scatter plot: each point is a passenger, colour shows whether they survived
plt.figure(figsize=(7, 5))
for label, colour, name in [(0, 'steelblue', 'Died'), (1, 'darkorange', 'Survived')]:
    mask = y_train == label
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colour, alpha=0.5, s=20, label=name)

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA: 2D View of Titanic Passengers')
plt.legend()
plt.tight_layout()
plt.show()


### Algorithm 4.10: Apriori (Association Rules)

**Type:** Unsupervised — Association Rule Mining
**Idea:** Discover which items tend to appear together.

Three key numbers:
- **Support:** How often does this combination appear? (P(A and B))
- **Confidence:** If A appears, how often does B also appear? (P(B | A))
- **Lift > 1:** A and B appear together more than you would expect by chance.

**Classic use case:** Supermarket basket analysis — customers who buy bread also buy butter.


In [ ]:
try:
    from mlxtend.frequent_patterns import apriori, association_rules
    from mlxtend.preprocessing import TransactionEncoder

    # 10 simple shopping baskets
    baskets = [
        ['milk', 'bread', 'butter'],
        ['beer', 'bread'],
        ['milk', 'bread', 'butter', 'eggs'],
        ['bread', 'butter'],
        ['milk', 'eggs'],
        ['beer', 'eggs'],
        ['milk', 'bread'],
        ['bread', 'butter', 'eggs'],
        ['milk', 'beer'],
        ['milk', 'bread', 'butter', 'beer'],
    ]

    # Convert to a True/False table (one row per basket, one column per item)
    encoder = TransactionEncoder()
    encoded_array = encoder.fit_transform(baskets)
    basket_df = pd.DataFrame(encoded_array, columns=encoder.columns_)

    # Find items that appear in at least 30% of baskets
    frequent_items = apriori(basket_df, min_support=0.3, use_colnames=True)

    # Generate association rules with lift > 1
    rules = association_rules(frequent_items, metric='lift', min_threshold=1.0)

    # Sort by lift (highest first) and show the top 8
    rules_sorted = rules.sort_values('lift', ascending=False)
    print("Top association rules (sorted by lift):")
    print(rules_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(8))

except ImportError:
    print("mlxtend is not installed. Run:  pip install mlxtend")


### Algorithm 4.11: LSTM (Long Short-Term Memory)

**Type:** Supervised — Deep Learning — Time Series
**Idea:** A special neural network that reads data in order (like reading a sentence word by word).
It has a "memory" that decides what to remember and what to forget.
**Use when:** The order of data matters: stock prices, weather, sensor readings, text.

We use a synthetic sine wave as our time series (simple and predictable).


In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense

    # 1) Create a simple noisy sine-wave dataset
    values = np.sin(np.linspace(0, 12, 260)) + 0.1 * np.random.randn(260)
    window = 10

    # 2) Build sequences: past 10 values -> next value
    X_list = []
    y_list = []
    for i in range(window, len(values)):
        X_list.append(values[i - window:i])
        y_list.append(values[i])

    X_ts = np.array(X_list).reshape(len(X_list), window, 1)
    y_ts = np.array(y_list)

    # 3) Split train/test in time order
    split = int(len(X_ts) * 0.8)
    X_ts_train = X_ts[:split]
    X_ts_test = X_ts[split:]
    y_ts_train = y_ts[:split]
    y_ts_test = y_ts[split:]

    # 4) Train a small LSTM model
    lstm_model = Sequential([
        LSTM(16, input_shape=(window, 1)),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')
    lstm_model.fit(X_ts_train, y_ts_train, epochs=5, batch_size=16, verbose=0)

    # 5) Predict and evaluate
    y_ts_predictions = lstm_model.predict(X_ts_test, verbose=0).flatten()
    print("LSTM — Test MSE:", round(mean_squared_error(y_ts_test, y_ts_predictions), 4))

    # 6) Plot
    plt.figure(figsize=(10, 3))
    plt.plot(y_ts_test, label='Actual')
    plt.plot(y_ts_predictions, label='LSTM Forecast', linestyle='--')
    plt.title('LSTM: Sine Wave Forecast')
    plt.legend()
    plt.tight_layout()
    plt.show()

except ImportError:
    print("TensorFlow is not installed. Run:  pip install tensorflow")


### Algorithm 4.12: Prophet

**Type:** Supervised — Time Series Forecasting
**Idea:** Prophet (by Meta/Facebook) breaks a time series into three parts:
- **Trend:** Is the overall value going up or down over time?
- **Seasonality:** Does it repeat every year, week, or day?
- **Holidays:** Does it spike or dip on special dates?

**Use when:** You have a business metric with clear seasonal patterns
(sales, website traffic, energy use).


In [ ]:
try:
    from prophet import Prophet

    # Create synthetic monthly sales data (48 months)
    dates = pd.date_range(start='2020-01-01', periods=48, freq='MS')
    trend = np.linspace(100, 160, 48)                   # slow upward trend
    seasonal = np.sin(np.linspace(0, 6 * 3.14159, 48)) * 20   # annual cycle
    noise = np.random.randn(48) * 5                     # random noise
    sales = trend + seasonal + noise

    # Prophet requires a DataFrame with exactly two columns: 'ds' and 'y'
    df_prophet = pd.DataFrame({
        'ds': dates,    # ds = date stamp
        'y': sales      # y = the value to forecast
    })

    # Create and train the model
    prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    prophet_model.fit(df_prophet)

    # Create a future DataFrame for the next 12 months
    future_dates = prophet_model.make_future_dataframe(periods=12, freq='MS')

    # Generate the forecast
    forecast = prophet_model.predict(future_dates)

    # Plot the result
    prophet_model.plot(forecast)
    plt.title('Prophet: Monthly Sales Forecast')
    plt.tight_layout()
    plt.show()

    # Show the last 12 rows (the future predictions)
    future_only = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)
    future_only.columns = ['Date', 'Forecast', 'Lower Bound', 'Upper Bound']
    print(future_only.to_string(index=False))

except ImportError:
    print("Prophet is not installed. Run:  pip install prophet")


## Section 5: Metrics and Evaluation

Choosing the right metric is as important as choosing the right model.

| Task | Key Metrics |
|------|------------|
| Regression | MSE, RMSE, MAE, R2 |
| Classification | Accuracy, Precision, Recall, F1-Score |
| Clustering | Inertia, Silhouette Score |

> **Curriculum link (Module 5)**


In [ ]:
# Regression metrics — using our Linear Regression predictions from earlier
print("=== Regression Metrics (California Housing) ===")
print()

mse  = mean_squared_error(y_h_test, y_h_predictions)
rmse = mse ** 0.5
mae  = mean_absolute_error(y_h_test, y_h_predictions)
r2   = r2_score(y_h_test, y_h_predictions)

print("MSE :", round(mse, 4),  "  (Mean Squared Error — penalises large errors heavily)")
print("RMSE:", round(rmse, 4), "  (Root MSE — same unit as the target, easier to interpret)")
print("MAE :", round(mae, 4),  "  (Mean Absolute Error — average error size)")
print("R2  :", round(r2, 4),   "  (1.0 = perfect, 0.0 = no better than predicting the mean)")


In [ ]:
# Classification metrics — using our Random Forest predictions
y_pred_rf = rf_model.predict(X_val)

print("=== Classification Metrics (Random Forest on Validation Set) ===")
print()

acc       = accuracy_score(y_val, y_pred_rf)
precision = precision_score(y_val, y_pred_rf)
recall    = recall_score(y_val, y_pred_rf)
f1        = f1_score(y_val, y_pred_rf)

print("Accuracy :", round(acc, 4),
      "  (What % of all predictions were correct?)")
print("Precision:", round(precision, 4),
      "  (Of everyone we predicted as Survived, what % actually survived?)")
print("Recall   :", round(recall, 4),
      "  (Of everyone who actually survived, what % did we catch?)")
print("F1-Score :", round(f1, 4),
      "  (Balanced average of Precision and Recall)")
print()
print(classification_report(y_val, y_pred_rf, target_names=['Died', 'Survived']))


In [ ]:
# A Confusion Matrix shows exactly what the model got right and wrong.
#
#                 Predicted: Died    Predicted: Survived
# Actual: Died       TN (good)          FP (model said survived, but they died)
# Actual: Survived   FN (missed!)       TP (good)

# --- Confusion matrix for Logistic Regression ---
cm_lr = confusion_matrix(y_val, log_reg.predict(X_val))
disp_lr = ConfusionMatrixDisplay(cm_lr, display_labels=['Died', 'Survived'])
disp_lr.plot(colorbar=False)
plt.title('Confusion Matrix: Logistic Regression')
plt.show()

# --- Confusion matrix for Random Forest ---
cm_rf = confusion_matrix(y_val, rf_model.predict(X_val))
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=['Died', 'Survived'])
disp_rf.plot(colorbar=False)
plt.title('Confusion Matrix: Random Forest')
plt.show()


In [ ]:
# ROC Curve: plots the True Positive Rate vs False Positive Rate at every threshold.
# AUC (Area Under the Curve): 1.0 = perfect model, 0.5 = random guessing.

plt.figure(figsize=(8, 6))

def add_roc_line(model, model_name):
    probs = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, probs)
    auc = roc_auc_score(y_val, probs)
    plt.plot(fpr, tpr, label=model_name + ' (AUC = ' + str(round(auc, 3)) + ')')

add_roc_line(log_reg, 'Logistic Regression')
add_roc_line(knn, 'KNN')
add_roc_line(dt_model, 'Decision Tree')
add_roc_line(rf_model, 'Random Forest')

# Diagonal line = random classifier
plt.plot([0, 1], [0, 1], color='black', linestyle='--', label='Random Classifier (AUC = 0.5)')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (Validation Set)')
plt.legend()
plt.tight_layout()
plt.show()


## Section 6: Advanced ML

> **Curriculum link (Module 6):** Three techniques to beat the bias-variance tradeoff.

| Technique | What it does |
|-----------|-------------|
| **Regularisation** | Adds a penalty for large weights — reduces overfitting |
| **Boosting** | Builds models one after another, each fixing the previous one's mistakes |
| **Bagging** | Builds many models in parallel on random subsets, then averages |


In [ ]:
# Regularisation: Ridge (L2 penalty) and Lasso (L1 penalty)
# These are like Linear Regression but with a penalty that keeps the weights small.
# Ridge: shrinks all weights toward zero
# Lasso: can set some weights to exactly zero (automatic feature selection)

# --- Ridge ---
ridge_model = Ridge(alpha=1.0)     # alpha controls how strong the penalty is
ridge_model.fit(X_h_train_scaled, y_h_train)
ridge_predictions = ridge_model.predict(X_h_test_scaled)
ridge_r2 = r2_score(y_h_test, ridge_predictions)

# --- Lasso ---
lasso_model = Lasso(alpha=0.01)
lasso_model.fit(X_h_train_scaled, y_h_train)
lasso_predictions = lasso_model.predict(X_h_test_scaled)
lasso_r2 = r2_score(y_h_test, lasso_predictions)

# --- Plain Linear Regression (for comparison) ---
plain_r2 = r2_score(y_h_test, y_h_predictions)

print("=== Regularisation Comparison (R2 on Test Set) ===")
print("Plain Linear Regression:", round(plain_r2, 4))
print("Ridge Regression:       ", round(ridge_r2, 4))
print("Lasso Regression:       ", round(lasso_r2, 4))
print()

# How many coefficients did Lasso set to zero?
zero_count = 0
for coef in lasso_model.coef_:
    if coef == 0:
        zero_count += 1
print("Lasso set", zero_count, "out of", len(lasso_model.coef_), "coefficients to zero.")
print("Those features were judged unimportant and removed automatically.")


In [ ]:
# Boosting: train models one by one, each one focuses on the mistakes of the previous.
# Gradient Boosting: uses the gradient of the loss to correct errors
# AdaBoost: gives more weight to the misclassified examples in each round

# --- Gradient Boosting ---
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train, y_train)
gb_predictions = gb_model.predict(X_val)
gb_accuracy = accuracy_score(y_val, gb_predictions)

# --- AdaBoost ---
ada_model = AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42)
ada_model.fit(X_train, y_train)
ada_predictions = ada_model.predict(X_val)
ada_accuracy = accuracy_score(y_val, ada_predictions)

print("=== Boosting (Validation Accuracy) ===")
print("Gradient Boosting:", round(gb_accuracy, 4))
print("AdaBoost:         ", round(ada_accuracy, 4))


## Section 7: Final Model Comparison

We now evaluate every classification model on the **test set**.
The test set has never been used before — this is the true measure of performance.


In [ ]:
# We test each model on X_test (unseen data) and record four metrics.
# Building the table row by row makes it easy to read.

results = []

all_models = [
    ('Logistic Regression', log_reg),
    ('KNN', knn),
    ('Naive Bayes', nb_model),
    ('SVM', svm_model),
    ('Decision Tree', dt_model),
    ('Random Forest', rf_model),
    ('Gradient Boosting', gb_model),
    ('AdaBoost', ada_model),
]

for model_name, model in all_models:
    preds = model.predict(X_test)
    results.append([
        model_name,
        round(accuracy_score(y_test, preds), 4),
        round(precision_score(y_test, preds), 4),
        round(recall_score(y_test, preds), 4),
        round(f1_score(y_test, preds), 4)
    ])

# Build the DataFrame and sort by F1-Score (best first)
columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
df_results = pd.DataFrame(results, columns=columns)
df_results = df_results.sort_values('F1-Score', ascending=False)
df_results = df_results.reset_index(drop=True)

print("=== Final Model Comparison (Test Set) ===")
print()
print(df_results.to_string(index=False))

# Bar chart
df_results.set_index('Model')[['Accuracy', 'F1-Score']].plot(kind='barh', figsize=(9, 5))
plt.title('Model Comparison: Accuracy and F1-Score (Test Set)')
plt.tight_layout()
plt.show()


## Congratulations - Lab 1 Complete!

Here is what you built today:

| Section | What you did |
|---------|-------------|
| EDA | Explored the Titanic dataset |
| Data Cleaning | Applied all 7 cleaning steps |
| 12 Algorithms | From Linear Regression to Prophet |
| Metrics | MSE, R2, accuracy, precision, recall, F1, confusion matrix, ROC-AUC |
| Advanced ML | Ridge, Lasso, Gradient Boosting, AdaBoost |
| Comparison | Ranked all models on the unseen test set |

> **Next:** Lab 2 — same pipeline, but you fill in the blanks.
